# MLP · tune an evidential PM₂.₅ regressor
**Stage 2 of 3:** run the split-and-save stage first. This notebook reads your saved `train.csv` (target `pm25_filled`) and `val.csv` (observed target `pm25`). It never opens `test.csv`. Stage 3 uses the winning settings in the matching usage notebook.

Three dense hidden layers process the selected inputs at each station and date.

The supplied CNN1D notebook's code uses one saved validation set, despite a docstring mentioning five-fold CV. Here the same held-out stations are used for every candidate; we call this **station-holdout validation**, not cross-validation. Scalers and tensors are prepared once and reused across trials. No values are filled on the validation stations.

### 1. Set the input and output directories
Install the project in editable mode with its Optuna extra before opening this notebook. The defaults are relative to the project root, whether you launch Jupyter there or from this notebook’s folder. Edit the settings or set the environment variables shown below. Run one notebook writer per study directory.

In [ ]:
%matplotlib inline
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pm25_edl.tuning import SCORES, load_best, run_study

here = Path.cwd().resolve()
ROOT = next((candidate for p in (here, *here.parents)
             for candidate in (p, p / 'educational')
             if (candidate / 'src' / 'pm25_edl').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from the unpacked project or one of its notebook directories.')
SPLITS = Path(os.environ.get('PM25_DATA_DIR', ROOT / 'data' / 'splits')).expanduser().resolve()
RUNS = Path(os.environ.get('PM25_RUN_DIR', ROOT / 'runs')).expanduser().resolve()

### 2. Choose the design and training budget
Choose `baseline` (total CMAQ), `at-bb` (three CMAQ components) or `at-bb-aod` (components plus AOD). WRF and static covariates stay fixed. The reference notebook uses 25 random trials, up to 100 epochs, patience 10 and seed 88. To test your installation quickly, reduce the budget and write to a **different** runs directory. A short run does not establish publication performance.

In [ ]:
MODEL = 'MLP'
FEATURE_OPTION = os.environ.get('PM25_FEATURE_OPTION', 'at-bb-aod')
RUN = RUNS / MODEL / FEATURE_OPTION
N_TRIALS = int(os.environ.get('PM25_N_TRIALS', 25))
MAX_EPOCHS = int(os.environ.get('PM25_MAX_EPOCHS', 100))
PATIENCE, BATCH_SIZE, SEED = 10, 64, 88
INTERVAL_LEVEL = 0.90
USE_COSINE_SIMILARITY = MODEL.startswith('TwoStage')  # Set False to disable the source penalty.

for filename, target in [('train.csv', 'pm25_filled'), ('val.csv', 'pm25')]:
    header = pd.read_csv(SPLITS / filename, nrows=0).columns
    if target not in header:
        raise ValueError(f'{filename} needs target {target!r}')
print('Split directory:', SPLITS, '\nStudy directory:', RUN)

### 3. Tune on observed validation labels
The three hidden widths form a non-increasing chain of powers of two. The range is 16–128; dropout is 0.0, 0.1, 0.2 or 0.3; Adam’s learning rate is drawn log-uniformly from $10^{-4}$ to $10^{-2}$. Only the CNN1D search was supplied; the other backbones reuse these ranges as educational choices. The evidential penalty weight (`evidential_weight`) is sampled from $10^{-4}$ to $10^{-1}$ on a log scale. For two-stage models with `USE_COSINE_SIMILARITY=True`, `lambda_sim` is sampled from 0.02 to 0.05 in steps of 0.01. The source penalty is zero for single-stage models or when the flag is False.

We minimize **mean Student-$t$ predictive negative log-likelihood (NLL)** in physical units, using the restored best validation epoch. At the restored validation-best model, a completed trial records $R^2$, RMSE, MAE, NLL and mean Winkler interval score separately for training and validation. Training scores use `pm25_filled` (including imputed labels); validation scores use only observed `pm25`. Both are computed in evaluation mode, without dropout. The Winkler score here uses a central 90% Student-$t$ predictive interval. Lower is better except for $R^2$, which is undefined for constant observations.

In [ ]:
study = run_study(SPLITS, RUN, model_name=MODEL, feature_option=FEATURE_OPTION,
                  n_trials=N_TRIALS, epochs=MAX_EPOCHS, patience=PATIENCE,
                  batch_size=BATCH_SIZE, seed=SEED, interval_level=INTERVAL_LEVEL,
                  use_cosine_similarity=USE_COSINE_SIMILARITY)
best = load_best(RUN, model_name=MODEL, feature_option=FEATURE_OPTION, split_dir=SPLITS)
print('Selected trial:', best['best_trial'], '| best epoch:', best['best_epoch'])
print('Resolved architecture:', best['model_kwargs'])
print('Training settings:', best['fit_kwargs'])

### 4. Compare trials and inspect the winning fit
The table includes `train_*` scores against filled training targets and the existing observed-validation scores. It remains sorted by validation NLL; training scores do not select the winning trial. Training loss includes the evidential penalty; the validation curve uses predictive NLL. The axes below use explicit epoch numbers. A joint study of all feature options would answer a different selection question; do not use the test set to choose among them.

In [ ]:
trials = pd.read_csv(RUN / 'trials.csv')
display(trials.loc[trials['state'].eq('COMPLETE'),
                   ['trial', 'best_epoch', 'train_n_obs', *['train_' + key for key in SCORES],
                    'n_obs', *SCORES]].sort_values('nll', ignore_index=True))
curves = study.best_trial.user_attrs
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
for ax, key, label in zip(axes, ['train_loss', 'val_nll'], ['Training objective', 'Validation NLL']):
    values = curves[key]
    ax.plot(np.arange(1, len(values) + 1), values, marker='.')
    ax.axvline(best['best_epoch'], color='tab:red', linestyle='--', label='Selected epoch')
    ax.set(xlabel='Epoch', ylabel=label)
    ax.legend()
plt.show()

### 5. Verify the saved result and the model contract
`study.sqlite3` retains individual Optuna trials; `trials.csv` is the portable table; `best.json` carries resolved widths and training parameters for Stage 3; `preprocessor.json` stores the train-fitted transforms. The manifest binds the study to the **contents** of the two input files and the settings and software used. The saved `study.sqlite3` is checked before starting: a matching rerun skips completed trials. Increase `N_TRIALS` from 25 to 50 in the same run directory to add only 25 new completed trials; it is a total budget, not an additional count. Use a new directory when changing data, code or training policy. Studies from before this search-space update are kept intact, but cannot be mixed with the new objective.

The final check uses a tiny mock tensor for verification only, independently of the research data.

In [ ]:
from pm25_edl.verification import check_model_contract

done = trials.loc[trials['state'].eq('COMPLETE')]
assert len(done) == N_TRIALS
assert done[['n_obs', 'rmse', 'mae', 'nll', 'mean_winkler']].notna().all().all()
assert done['n_obs'].gt(0).all() and 'r2' in done
assert done[['train_n_obs', 'train_rmse', 'train_mae', 'train_nll', 'train_mean_winkler']].notna().all().all()
assert done['train_n_obs'].gt(0).all() and 'train_r2' in done
assert np.isclose(best['validation_scores']['nll'], done['nll'].min())
assert all((RUN / f).is_file() for f in ['study.sqlite3', 'manifest.json', 'preprocessor.json', 'best.json'])
check_model_contract(MODEL)
print('Verified saved training/validation results and the mock model contract.')